In [1]:
import snowflake.connector
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas

# ── FILLING IN DETAILS  ────────────────────────────────────
SNOWFLAKE_ACCOUNT  = 'kaejkcm-ql87782'   
SNOWFLAKE_USER     = 'DEBOJYOTIBAASU'         
SNOWFLAKE_PASSWORD = 'Debojyoti@Basu@7044'      
# ──────────────────────────────────────────────────────────────────

conn = snowflake.connector.connect(
    account   = SNOWFLAKE_ACCOUNT,
    user      = SNOWFLAKE_USER,
    password  = SNOWFLAKE_PASSWORD,
    warehouse = 'SUPPLY_WH',
    database  = 'SUPPLY_CHAIN_DB',
    schema    = 'RAW'
)

print("✅ Connected to Snowflake successfully!")
print(f"   Account  : {SNOWFLAKE_ACCOUNT}")
print(f"   Database : SUPPLY_CHAIN_DB")
print(f"   Schema   : RAW")

✅ Connected to Snowflake successfully!
   Account  : kaejkcm-ql87782
   Database : SUPPLY_CHAIN_DB
   Schema   : RAW


In [3]:
import os

# Point data folder
DATA_PATH = r'C:\Users\Debojyoti (DJ)\supply_chain_data'

# Loading all 3 tables
print("Loading CSV files...")

dim_sku = pd.read_csv(os.path.join(DATA_PATH, 'DIM_SKU.csv'))
print(f"✅ DIM_SKU loaded       : {dim_sku.shape[0]:,} rows")

dim_date = pd.read_csv(os.path.join(DATA_PATH, 'DIM_DATE.csv'))
print(f"✅ DIM_DATE loaded      : {dim_date.shape[0]:,} rows")

fact_demand = pd.read_csv(os.path.join(DATA_PATH, 'FACT_DEMAND.csv'))
print(f"✅ FACT_DEMAND loaded   : {fact_demand.shape[0]:,} rows")

print(f"\n📊 Total records loaded : {dim_sku.shape[0] + dim_date.shape[0] + fact_demand.shape[0]:,}")

Loading CSV files...
✅ DIM_SKU loaded       : 50,000 rows
✅ DIM_DATE loaded      : 1,096 rows
✅ FACT_DEMAND loaded   : 12,244,320 rows

📊 Total records loaded : 12,295,416


In [5]:
# ── CREATING RAW TABLES IN SNOWFLAKE ────────────────────────────────

cursor = conn.cursor()

# Creating DIM_SKU table
cursor.execute("""
    CREATE OR REPLACE TABLE SUPPLY_CHAIN_DB.RAW.DIM_SKU (
        SKU_ID          VARCHAR(20),
        SKU_NAME        VARCHAR(100),
        CATEGORY        VARCHAR(50),
        SUPPLIER        VARCHAR(100),
        ABC_CLASS       VARCHAR(1),
        XYZ_CLASS       VARCHAR(1),
        COMBINED_CLASS  VARCHAR(2),
        UNIT_COST_USD   FLOAT,
        LEAD_TIME_DAYS  INT,
        SAFETY_STOCK_DAYS INT,
        IS_ACTIVE       BOOLEAN,
        CREATED_DATE    DATE
    )
""")
print("✅ RAW.DIM_SKU table created")

# Creating DIM_DATE table
cursor.execute("""
    CREATE OR REPLACE TABLE SUPPLY_CHAIN_DB.RAW.DIM_DATE (
        DATE_KEY        DATE,
        YEAR            INT,
        QUARTER         INT,
        MONTH           INT,
        MONTH_NAME      VARCHAR(20),
        WEEK            INT,
        DAY_OF_WEEK     INT,
        DAY_NAME        VARCHAR(20),
        IS_WEEKEND      BOOLEAN,
        IS_MONTH_END    BOOLEAN,
        IS_QUARTER_END  BOOLEAN,
        FISCAL_YEAR     INT,
        FISCAL_QUARTER  INT
    )
""")
print("✅ RAW.DIM_DATE table created")

# Creating FACT_DEMAND table
cursor.execute("""
    CREATE OR REPLACE TABLE SUPPLY_CHAIN_DB.RAW.FACT_DEMAND (
        DEMAND_DATE       DATE,
        SKU_ID            VARCHAR(20),
        REGION            VARCHAR(50),
        ACTUAL_DEMAND     INT,
        FORECAST_DEMAND   INT,
        VARIANCE          INT,
        UNIT_COST_USD     FLOAT,
        DEMAND_VALUE_USD  FLOAT,
        ABC_CLASS         VARCHAR(1),
        XYZ_CLASS         VARCHAR(1),
        SUPPLIER          VARCHAR(100),
        CATEGORY          VARCHAR(50)
    )
""")
print("✅ RAW.FACT_DEMAND table created")

print("\n All 3 tables created in Snowflake RAW schema")

✅ RAW.DIM_SKU table created
✅ RAW.DIM_DATE table created
✅ RAW.FACT_DEMAND table created

 All 3 tables created in Snowflake RAW schema


In [7]:
# ── UPLOADING SMALL TABLES FIRST ─────────────────────────────────────

# Fixing column names to uppercase 
dim_sku.columns = [col.upper() for col in dim_sku.columns]
dim_date.columns = [col.upper() for col in dim_date.columns]

# Upload -  DIM_SKU
print("Uploading DIM_SKU... ")
success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=dim_sku,
    table_name='DIM_SKU',
    database='SUPPLY_CHAIN_DB',
    schema='RAW'
)
print(f"✅ DIM_SKU uploaded — {nrows:,} rows in {nchunks} chunks")

# Upload - DIM_DATE
print("\nUploading DIM_DATE...")
success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=dim_date,
    table_name='DIM_DATE',
    database='SUPPLY_CHAIN_DB',
    schema='RAW'
)
print(f"✅ DIM_DATE uploaded — {nrows:,} rows in {nchunks} chunks")

Uploading DIM_SKU... 
✅ DIM_SKU uploaded — 50,000 rows in 1 chunks

Uploading DIM_DATE...
✅ DIM_DATE uploaded — 1,096 rows in 1 chunks


In [9]:
# ── UPLOADING FACT_DEMAND (12 MILLION ROWS) ──────────────────────────
# We upload in chunks so it doesn't crash


import time

fact_demand.columns = [col.upper() for col in fact_demand.columns]

print("Uploading FACT_DEMAND — 12 million rows...")


start_time = time.time()

success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=fact_demand,
    table_name='FACT_DEMAND',
    database='SUPPLY_CHAIN_DB',
    schema='RAW',
    chunk_size=200000    # uploads 200k rows at a time
)

end_time = time.time()
minutes = round((end_time - start_time) / 60, 1)

print(f"✅ FACT_DEMAND uploaded!")
print(f"   Rows uploaded : {nrows:,}")
print(f"   Chunks used   : {nchunks}")
print(f"   Time taken    : {minutes} minutes")
print(f"\n ALL DATA IS NOW LIVE IN SNOWFLAKE")

Uploading FACT_DEMAND — 12 million rows...
✅ FACT_DEMAND uploaded!
   Rows uploaded : 12,244,320
   Chunks used   : 62
   Time taken    : 1.2 minutes

 ALL DATA IS NOW LIVE IN SNOWFLAKE


In [11]:
# ── VERIFYING ROW COUNTS IN SNOWFLAKE ────────────────────────────────

cursor = conn.cursor()

tables = ['DIM_SKU', 'DIM_DATE', 'FACT_DEMAND']

print("Verifying row counts in Snowflake...\n")
print(f"{'Table':<20} {'Rows in Snowflake':>20}")
print("-" * 42)

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM SUPPLY_CHAIN_DB.RAW.{table}")
    count = cursor.fetchone()[0]
    print(f"{table:<20} {count:>20,}")

print("-" * 42)
print("\n✅ Verification complete!")
print(" Phase 3 — Data Loading INTO SNOWFLAKE DONE")

Verifying row counts in Snowflake...

Table                   Rows in Snowflake
------------------------------------------
DIM_SKU                            50,000
DIM_DATE                            1,096
FACT_DEMAND                    12,244,320
------------------------------------------

✅ Verification complete!
 Phase 3 — Data Loading INTO SNOWFLAKE DONE
